***Create and Inspect a Tensor***

In [2]:
import torch

def make_tensor():
	"""Return a 2x3 float32 tensor [[1, 2, 3], [4, 5, 6]]."""
	# TODO
	return torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

t = make_tensor()
print(t)
print("----------------")
t = make_tensor()
print(tuple(t.shape))
print("----------------")
t = make_tensor()
print(float(t[1, 2].item()))

tensor([[1., 2., 3.],
        [4., 5., 6.]])
----------------
(2, 3)
----------------
6.0


***Reshape and Transpose a Tensor***

In [3]:
import torch

def reshape_transpose(t):
	"""Reshape a 1D tensor of 6 elements to 2x3 (row-major) and return its transpose.

	Args:
		t (torch.Tensor): 1D tensor with exactly 6 elements.

	Returns:
		torch.Tensor: Transpose of the 2x3 reshape, with shape (3, 2).
	"""
	# TODO: reshape to (2, 3) then transpose to (3, 2)
	return t.reshape(2, 3).T   # or use .t() 

print(reshape_transpose(torch.tensor([1, 2, 3, 4, 5, 6])))
print("---------------------")
print(reshape_transpose(torch.tensor([1., 2., 3., 4., 5., 6.])))


tensor([[1, 4],
        [2, 5],
        [3, 6]])
---------------------
tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])


***Gradient of a Square with Autograd***

In [4]:
import torch

def grad_of_square(x_val):
	
	# TODO: create tensor, compute y = x**2, backward, return grad
	x = torch.tensor(x_val, requires_grad=True)
	y = x ** 2
	y.backward()
	return x.grad.item()
  

print(grad_of_square(3.0))

6.0


***Gradient of a Weighted Sum of Squares***

In [5]:
import torch

def grad_wss(w_list, x_list):
	"""Build w (requires_grad) and x from lists, compute
	loss = 0.5 * sum((w * x)**2), backward, return w.grad
	as a list of floats rounded to 4 decimals.
	"""
	# TODO
	w = torch.tensor(w_list, requires_grad=True)
	x = torch.tensor(x_list)
	L = 1/2 * ((w * x)**2).sum()
	L.backward()
	return [round(g, 4) for g in w.grad.tolist()]

print(grad_wss([1.0, 2.0], [3.0, 4.0]))

[9.0, 32.0]


***Lab : Fit Linear Regression with Autograd***

In [6]:
import torch
import numpy as np


def fit_linear_regression(X, y, lr=0.1, steps=500):
	
	w = torch.zeros(X.shape[1], requires_grad=True)
	b = torch.tensor(0.0, requires_grad=True)

	for _ in range(steps):
		y_pred = X @ w + b 
		loss = torch.mean((y_pred - y)**2)
		loss.backward()
		# manual GD update under torch.no_grad()
		with torch.no_grad():
			w -= lr * w.grad
			b -= lr * b.grad
		# zero gradients
		w.grad.zero_()
		b.grad.zero_()

	# return detached w, b
	return w.detach(), b.detach()
	


***Single Linear Neuron Forward***

In [7]:
import torch
import torch.nn as nn

def single_neuron_forward(x):
	# create the layer
	neuron = nn.Linear(3, 1)
	
	# fix the weights inside torch.no_grad()
	with torch.no_grad():
		neuron.weight.copy_(torch.tensor([[0.5, -0.2, 0.3]]))
		neuron.bias.copy_(torch.tensor([0.1]))
	
	# run forward and return as float
	return float(neuron(x).item())

print(single_neuron_forward(torch.tensor([[1.0, 1.0, 1.0]])))
print("----------------------------")
print(single_neuron_forward(torch.tensor([[2.0, 0.0, 0.0]])))



0.7000000476837158
----------------------------
1.100000023841858


***Two-Layer MLP Forward Pass***

In [8]:
import torch
import torch.nn as nn


def two_layer_mlp_forward(x, w1, b1, w2, b2):

	model = nn.Sequential(
		nn.Linear(2, 2), # index 0
		nn.ReLU(),       # index 1
		nn.Linear(2, 1)  # index 2 
	)

	with torch.no_grad():
		model[0].weight.copy_(w1)  
		model[0].bias.copy_(b1)

		model[2].weight.copy_(w2)
		model[2].bias.copy_(b2)

	out = model(x)
	return out.item()

x = torch.tensor([[1.0, -1.0]])
w1 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b1 = torch.tensor([0.0, 0.0])
w2 = torch.tensor([[1.0, 1.0]])
b2 = torch.tensor([0.0])
print(two_layer_mlp_forward(x, w1, b1, w2, b2))

1.0


***Implement ReLU and Leaky ReLU***

In [9]:
import torch

def relu(t):
	"""Element-wise ReLU: max(0, t).

	Args:
		t (torch.Tensor): input tensor

	Returns:
		torch.Tensor: activated tensor
	"""
	# TODO: implement with pure torch ops
	return torch.clamp(t, min=0)

def leaky_relu(t, slope=0.01):
	"""Element-wise Leaky ReLU with given negative slope.

	Args:
		t (torch.Tensor): input tensor
		slope (float): slope for negative values

	Returns:
		torch.Tensor: activated tensor
	"""
	# TODO: implement with pure torch ops
	return torch.where(t > 0, t, slope * t)


***Numerically Stable Softmax***

In [10]:
import torch

def softmax(t, dim):
	"""Numerically stable softmax along dim.

	Args:
		t (torch.Tensor): input tensor
		dim (int): dimension along which to apply softmax

	Returns:
		torch.Tensor: tensor of same shape as t; slices along dim sum to 1
	"""
	# TODO: subtract max along dim, exp, then normalize
	max_val = t.max(dim = dim, keepdim = True).values
	shifted = t - max_val
	exp = torch.exp(shifted)
	sum_exp = exp.sum(dim = dim, keepdim= True)
	return exp / sum_exp

t = torch.tensor([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]])
result = softmax(t, dim=1)
print(torch.round(result * 1e4) / 1e4)
print("-------------")
t = torch.tensor([[1000.0, 1001.0, 1002.0]])
result = softmax(t, dim=1)
print(torch.round(result * 1e4) / 1e4)

tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])
-------------
tensor([[0.0900, 0.2447, 0.6652]])


***Mean Squared Error from Scratch***

In [11]:
import torch

def mse(pred, target):
	"""
	Compute mean squared error between pred and target.

	Args:
		pred (torch.Tensor): Predicted values.
		target (torch.Tensor): Ground-truth values (same shape as pred).

	Returns:
		float: Mean of squared differences.
	"""
	# TODO
	mse = torch.mean((pred - target)**2)
	return  mse.item()

print(mse(torch.tensor([0.0, 0.0]), torch.tensor([1.0, 0.0])))
print(mse(torch.tensor([5.0]), torch.tensor([2.0])))



0.5
9.0


***Binary Cross-Entropy from Logits***

In [12]:
import torch

def bce_with_logits(logits, targets):
	loss = (
		torch.clamp(logits, min=0)
		- logits * targets
		+ torch.log(1 + torch.exp(-torch.abs(logits)))
	)

	loss = torch.mean(loss)
	return round(loss.item(), 4)

print(bce_with_logits(torch.tensor([2.0, -2.0]), torch.tensor([1.0, 0.0])))


0.1269


***Dropout in Train vs Eval Mode***

In [13]:
import torch
import torch.nn as nn


def dropout_demo():
	"""Demonstrate Dropout behavior in eval vs train mode.

	Returns:
		tuple: (eval_output, train_nonzero_count)
			eval_output: result of Dropout(ones) in eval mode (identity)
			train_nonzero_count: int count of nonzero elements after Dropout in train mode
	"""
	# TODO: seed, ones(10), Dropout(0.5), eval output, train nonzero count
	torch.manual_seed(0)
	X = torch.ones(10)

	drop = nn.Dropout(p = 0.5)

	# Evaluation mode 
	drop.eval()
	eval_output = drop(X)

	# Training mode 
	drop.train()
	train_output = drop(X)

	# Count 
	count_nonzeros = torch.count_nonzero(train_output).item()

	return eval_output , count_nonzeros

print(dropout_demo()[1])
print("----------")
out, cnt = dropout_demo()
print(tuple(out.shape))
print(out.dtype)
print(type(cnt).__name__)

4
----------
(10,)
torch.float32
int


***BatchNorm1d Forward in Eval Mode***

In [14]:
import torch

def bn_eval(x, mean, var, gamma, beta, eps=1e-5):
	"""Apply batch-norm inference normalization.

	Args:
		x (Tensor): input tensor
		mean (Tensor): running mean
		var (Tensor): running variance
		gamma (Tensor): scale parameter
		beta (Tensor): shift parameter
		eps (float): numerical stability constant

	Returns:
		Tensor: normalized and affine-transformed tensor
	"""
	# TODO
	normalized =( x - mean ) /  (torch.sqrt(var + eps))
	out = gamma * normalized + beta
	return out

print(torch.round(bn_eval(torch.tensor([3.0, 7.0]), torch.tensor(5.0), torch.tensor(4.0), torch.tensor(3.0), torch.tensor(-1.0))*1e4)/1e4)

tensor([-4.,  2.])


***Count Parameters of a Sequential Model***

In [15]:
import torch
import torch.nn as nn

def count_params():
	"""Build Sequential(Linear(4,8), ReLU, Linear(8,2)) and return trainable param count.

	Returns:
		int: total number of trainable parameters
	"""
	# TODO: build the model and sum p.numel() for trainable params
	model = nn.Sequential(
		nn.Linear(4, 8),  # weight.shape = (4, 8) and bias.shape = (8, ) || total shape = 4*8 + 8 = 32 + 8 = 40
		nn.ReLU(),
		nn.Linear(8, 2)   # weight.shape = (8, 2) and bias.shape = (2, ) || total shape = 8*2 + 2 = 16 + 2 = 18 
	)

	total = 0 

	for p in model.parameters():
		if p.requires_grad :
			total += p.numel()
	return total

print(count_params())

58


***Lab : MLP with Dropout and BatchNorm***

In [2]:
import torch
import torch.nn as nn


class RegularizedMLP(nn.Module):
    """MLP with BatchNorm1d and Dropout for binary classification."""

    def __init__(self, input_dim: int, hidden_dim: int = 64, dropout_p: float = 0.3):
        super().__init__()

        self.network = nn.Sequential(
            # Hidden Block 1
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            # Hidden Block 2
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            # Output Layer
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return shape (N,) logits for batch x of shape (N, input_dim)."""
        return self.network(x).squeeze(-1)


def train_model(model, X_train, y_train, epochs=150, lr=1e-2):
    """Train model in-place with BCEWithLogitsLoss + Adam. Return model."""

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()

    for _ in range(epochs):
        optimizer.zero_grad()

        logits = model(X_train)
        loss = criterion(logits, y_train)

        loss.backward()
        optimizer.step()

    return model

***Conv2d Output Shape***

In [1]:
def conv_out_shape(h, w, kernel, stride, padding):
    # // Floor Division 
    h_out = (h + 2 * padding - kernel) // stride + 1
    w_out = (w + 2 * padding - kernel) // stride + 1

    return (h_out, w_out)

print(conv_out_shape(28, 28, 5, 2, 0))
print('---------------')
print(conv_out_shape(16, 32, 3, 2, 1))

(12, 12)
---------------
(8, 16)


***Apply a 2D Convolution***

In [1]:
import torch
import torch.nn as nn


def apply_conv2d():
    """Build nn.Conv2d(1, 1, kernel_size=2, bias=False), set a fixed kernel, convolve a fixed input.

    Under torch.no_grad(), set weight to tensor([[[[1.0, 0.0], [0.0, 1.0]]]]).
    Input is tensor([[[[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0],
                       [7.0, 8.0, 9.0]]]]).

    Returns:
        torch.Tensor: Output of shape (1, 1, 2, 2).
    """
    conv = nn.Conv2d(1, 1, kernel_size=2, bias=False)

    x = torch.tensor([[[[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0],
                        [7.0, 8.0, 9.0]]]])

    with torch.no_grad():
        conv.weight.copy_(torch.tensor([[[[1.0, 0.0],
                                          [0.0, 1.0]]]]))
        output = conv(x)

    return output

import os as _os
_dn = _os.open(_os.devnull, _os.O_WRONLY); _old = _os.dup(2); _os.dup2(_dn, 2)
try:
    print(apply_conv2d()[0, 0, 0, 0].item())
finally:
    _os.dup2(_old, 2); _os.close(_old); _os.close(_dn)

6.0


***Lab : Train a Tiny CNN Image Classifier***

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


class TinyCNN(nn.Module):
    """Small CNN: Conv2d -> ReLU -> pool -> (optional extras) -> Linear."""

    def __init__(self, img_size=8, n_classes=2):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(kernel_size=2)

        self.classifier = nn.Linear(
            8 * (img_size // 2) * (img_size // 2),
            n_classes
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)

        logits = self.classifier(x)

        return logits


def build_model(img_size=8, n_classes=2):
    """Return an instance of your TinyCNN (or equivalent nn.Module)."""
    return TinyCNN(
        img_size=img_size,
        n_classes=n_classes
    )


def train_model(model, train_x, train_y, epochs=15, lr=0.01,
                batch_size=32, seed=0):
    """Train model on train_x/train_y and return the trained model."""

    # Make shuffling deterministic
    torch.manual_seed(seed)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    model.train()

    n = train_x.shape[0]

    for epoch in range(epochs):

        # Shuffle training examples
        indices = torch.randperm(n)

        for start in range(0, n, batch_size):

            batch_indices = indices[start:start + batch_size]

            x_batch = train_x[batch_indices]
            y_batch = train_y[batch_indices]

            # 1. Clear old gradients
            optimizer.zero_grad()

            # 2. Forward pass
            logits = model(x_batch)

            # 3. Compute loss
            loss = criterion(logits, y_batch)

            # 4. Backpropagation
            loss.backward()

            # 5. Update weights
            optimizer.step()

    return model

***One SGD Update Step***

In [2]:
import torch

def sgd_step(w, grad, lr):
    """Perform one SGD update step.

    Args:
        w: Current parameter tensor.
        grad: Gradient tensor (same shape as w).
        lr: Learning rate (float).

    Returns:
        Updated parameter tensor w - lr * grad.
    """
    return w - torch.mul(lr , grad)

print(sgd_step(torch.tensor([[1.0, 2.0], [3.0, 4.0]]), torch.tensor([[0.1, 0.2], [0.3, 0.4]]), 1.0))

tensor([[0.9000, 1.8000],
        [2.7000, 3.6000]])


***SGD with Momentum Step***

In [3]:
import torch

def momentum_step(w, grad, v, lr, mu):
    """One SGD-with-momentum step.

    Args:
        w: parameter tensor
        grad: gradient tensor (same shape as w)
        v: velocity tensor (same shape as w)
        lr: learning rate (float)
        mu: momentum coefficient (float)

    Returns:
        (w_new, v_new) tuple of tensors
    """
    # TODO: v_new = mu * v + grad; w_new = w - lr * v_new
    v_new = torch.mul(mu , v) + grad
    w_new = w - torch.mul(lr , v_new)
    return v_new , w_new

w = torch.tensor([3.0])
grad = torch.tensor([1.0])
v = torch.tensor([2.0])
print(momentum_step(w, grad, v, 0.5, 0.5))

(tensor([2.]), tensor([2.]))


***One Adam Update Step***

In [2]:
import torch

def adam_step(w, grad, m, v, t, lr, beta1, beta2, eps):
    """One Adam update with bias-corrected moments.

    Args:
        w: current parameters (torch.Tensor)
        grad: gradient of the loss w.r.t. w (torch.Tensor)
        m: first moment estimate (torch.Tensor)
        v: second moment estimate (torch.Tensor)
        t: timestep, 1-indexed (int)
        lr: learning rate (float)
        beta1: exp. decay for first moment (float)
        beta2: exp. decay for second moment (float)
        eps: numerical stability constant (float)

    Returns:
        Tuple (w_new, m_new, v_new) as torch.Tensor values.
    """
    # Update first moment (EMA of gradients)
    m_new = beta1 * m + (1 - beta1) * grad

    # Update second moment (EMA of squared gradients)
    v_new = beta2 * v + (1 - beta2) * (grad * grad)

    # Bias correction
    m_hat = m_new / (1 - beta1 ** t)
    v_hat = v_new / (1 - beta2 ** t)

    # Parameter update
    w_new = w - lr * m_hat / (torch.sqrt(v_hat) + eps)

    return w_new, m_new, v_new

w_new, _, _ = adam_step(torch.tensor([1.0, 0.0, -1.0]), torch.tensor([0.5, 0.5, 0.5]), torch.zeros(3), torch.zeros(3), 1, 0.01, 0.5, 0.5, 1e-8)
print(torch.round(w_new * 1e5) / 1e5)

tensor([ 0.9900, -0.0100, -1.0100])


***Lab : Design Your Own Pytorch Optimizer***

In [ ]:
import math
import torch
from torch.optim.optimizer import Optimizer

class MyOptimizer(Optimizer):
    """
    Design your own optimizer!
    - You can base it on SGD, RMSProp, Adam, or create something new.
    - Must subclass torch.optim.Optimizer.
    - Only dense gradients are supported.
    """
    def __init__(self, params, lr=1e-3):
        # You can add your own hyperparameters here
        defaults = dict(lr=lr)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                # --- TODO: implement your own update rule ---
                # Example: simple SGD update
                # p.add_(grad, alpha=-lr)
                pass

        return loss


***Batch a TensorDataset with DataLoader***

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def batch_stats(X, y):
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=4, shuffle=False)

    num_batches = 0
    first_batch_X_shape = None

    for batch_X, batch_y in loader:
        num_batches += 1

        if first_batch_X_shape is None:
            first_batch_X_shape = tuple(batch_X.shape)

    return num_batches, first_batch_X_shape

X = torch.zeros(12, 3)
y = torch.ones(12, dtype=torch.long)
print(batch_stats(X, y))

(3, (4, 3))


***Lab : Pytorch DataLoader***

In [ ]:
import torch

class MyTransform:
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (1, 28, 28) float tensor in [0,1]
        Return: transformed tensor, same shape/dtype.
        Must be non-identity and deterministic.
        """
        # TODO: implement your custom transformation logic here

        # Change contrast and brightness
        x = x * 1.2 + 0.05

        # Keep values bounded
        x = x.clamp(0.0, 1.0)

        return x


***Count Parameters of a Sequential Model***

In [16]:
import torch
import torch.nn as nn

def count_params():
	"""Build Sequential(Linear(4,8), ReLU, Linear(8,2)) and return trainable param count.

	Returns:
		int: total number of trainable parameters
	"""
	# TODO: build the model and sum p.numel() for trainable params
	model = nn.Sequential(
		nn.Linear(4, 8),  # weight.shape = (4, 8) and bias.shape = (8, ) || total shape = 4*8 + 8 = 32 + 8 = 40
		nn.ReLU(),
		nn.Linear(8, 2)   # weight.shape = (8, 2) and bias.shape = (2, ) || total shape = 8*2 + 2 = 16 + 2 = 18 
	)

	total = 0 

	for p in model.parameters():
		if p.requires_grad :
			total += p.numel()
	return total

print(count_params())

58


***One Training Step***

In [2]:
import torch

def train_step(model, x, y, optimizer, loss_fn):
    """Run one training step and return the pre-update loss as a float.

    Args:
        model: torch.nn.Module to train.
        x: Input batch tensor.
        y: Target batch tensor.
        optimizer: torch.optim optimizer bound to model parameters.
        loss_fn: Callable (pred, y) -> scalar loss tensor.

    Returns:
        float: Loss value computed before optimizer.step().
    """
    # TODO: zero_grad -> forward -> loss -> backward -> step; return float loss
    optimizer.zero_grad()
    # Forward
    pred = model(x)
    # Loss
    loss = loss_fn(pred, y)
    # Bachward
    loss.backward()
    # Step
    optimizer.step()
    return float(loss.item()) 

import torch
import torch.nn as nn
model = nn.Linear(2, 1)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[1.0, 1.0]]))
    model.bias.copy_(torch.tensor([0.0]))
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0, 1.0]])
y = torch.tensor([[0.0]])
print(round(train_step(model, x, y, opt, loss_fn), 4))

4.0
